# 03 — Diagnóstico: por que a reflexão não bate o baseline (ARC, validação)

Motivação (24/08/2026 em diante): a etapa de avaliação (`results/eval/*`) vem mostrando
reflexão perdendo até para o baseline mais simples, sem reflexão nenhuma. Este notebook
existe para investigar a causa, não para reproduzir o paper (isso já está em `analysis.ipynb`
— ver [[reflection_mcq_analysis_notebook]] na memória do projeto).

**Pergunta central:** existe alguma combinação de `k` (quantas reflexões recuperar) e de um
limiar mínimo de similaridade que faz a reflexão superar o baseline? Se sim, em que faixa? Se
não, isso é evidência de que a reflexão recuperada por similaridade não está ajudando — mesmo
nos casos "fáceis" (questão de validação muito parecida com uma de treino).

**Desenho deste notebook:**

1. Dataset: **ARC, split de validação** (298 itens, nunca usado no pipeline principal — só
   `train` e `test` têm baseline gerado hoje). Fica em `data/processed/arc/validation.jsonl`
   e é idêntico ao `data/splits/arc/validation.jsonl` que o pacote `rmcq` lê.
2. Modelos: **phi4-mini** e **llama3-8b** (os dois alunos ativos em `rmcq.config`).
   Reaproveitamos as reflexões que cada um já escreveu sobre si mesmo (`self_reflection`) na
   etapa de treino — `results/reflections/{modelo}__{modelo}__{simple,complex}/arc.jsonl`,
   286 itens cada, já prontos em disco. Nenhuma reflexão nova é gerada aqui.
3. Baseline novo: os dois modelos respondem a validação **sem nenhuma reflexão** — isso não
   existia ainda (só train/test) e é o ponto de comparação de tudo o que vem depois.
4. Similaridade treino↔validação: embeddings com o mesmo modelo do pipeline
   (`BAAI/bge-large-en-v1.5`), reaproveitando o embedding de treino já cacheado no índice
   oficial quando possível. A partir da distribuição de similaridade top-1, escolhemos valores
   de `k` e de limiar de similaridade para testar — isso é decidido a partir dos dados, não
   chutado.
5. Grade `k × threshold × depth`, por modelo: para cada questão de validação, recupera até `k`
   reflexões de treino com similaridade ≥ threshold (rastreando quantas foram *de fato*
   recuperadas — pode ser menos que `k`) e, se **nenhuma** passar do limiar, cai no prompt de
   baseline puro, sem reflexão. Grava tudo em `results/diagnostics/`, de forma retomável.
6. Consolidação: acurácia e **reflection utility** (métrica já definida no projeto —
   errado→certo menos certo→errado, ver `rmcq.stages.analyze.utility`) de cada configuração
   contra o baseline. Tabela dos casos em que a reflexão ganhou, e gráficos.

**Escopo deliberadamente fora deste notebook** (fácil de estender, ver comentários no código):
reflexão externa (professor ≠ aluno), outros datasets além do ARC, professores Azure/gpt-5.

**Todos os prompts usados aqui são cópias locais, editáveis, dos templates de
`rmcq/common.py`** — não são importados como função opaca. Uma célula de verificação compara
com o original a cada execução e avisa (sem travar) se algo divergiu.

**Custo:** a célula de execução da grade carrega os dois modelos via o backend configurado em
`RMCQ_BACKEND` (vLLM por padrão) e gera uma resposta por item por configuração. Use
`SMOKE_TEST = True` primeiro para validar a canalização inteira em segundos, com o
`StubBackend` (sem GPU) ou com um `limit` pequeno no backend real.


## 0. Setup

In [ ]:
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # força o uso da GPU 0 (ou seja, a primeira)
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pacote rmcq

import rmcq  # carrega o .env antes de qualquer import de torch/transformers/vllm
print(rmcq.env_summary())

from rmcq.backends import GenParams, get_backend
from rmcq.common import make_record, read_jsonl, write_jsonl  # utilitários, não prompts
from rmcq.config import (
    EMBED_BATCH_SIZE, EMBEDDER, HF_HOME, INDEX_DIR, RESULTS_DIR, SEED, STUDENT_GEN,
    hf_token,
)
from rmcq.data import baseline_path, index_paths, load_split
from rmcq.stages.analyze import SIM_BINS, accuracy_block, transferability, utility
from rmcq.store import JsonlStore, Timer, get_logger, progress

log = get_logger(__name__)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (7, 4)

DATASET = "arc"
STUDENTS = ["phi4-mini", "llama3-8b"]   # alunos ativos em rmcq.config.ACTIVE_MODELS
DEPTHS = ["simple", "complex"]           # as duas profundidades de reflexão já geradas

# --- controles de execução ---------------------------------------------------
# Rode primeiro com SMOKE_TEST = True: troca o backend real pelo StubBackend
# (determinístico, sem GPU, ver rmcq/backends/stub.py) e limita a poucos itens.
# Serve para validar a canalização inteira (prompts, recuperação, gravação,
# consolidação, gráficos) em segundos antes de gastar horas de GPU de verdade.
SMOKE_TEST = False
SMOKE_LIMIT = 8

BACKEND_KIND = "stub" if SMOKE_TEST else None   # None = usa RMCQ_BACKEND do .env (vllm)
RUN_LIMIT = SMOKE_LIMIT if SMOKE_TEST else None

print(f"dataset={DATASET}  students={STUDENTS}  depths={DEPTHS}")
print(f"SMOKE_TEST={SMOKE_TEST}  backend_kind={BACKEND_KIND!r}  limit={RUN_LIMIT}")


## 1. Dataset — ARC, split de validação

`rmcq.data.load_split` lê de `data/splits/arc/validation.jsonl`. Conferimos que é o mesmo
conteúdo do `data/processed/arc/validation.jsonl` citado no pedido original (é — os splits
finais são copiados de `processed/` por `01_formatacao_e_selecao.ipynb`, sem the ressample
adicional que train sofreu).


In [ ]:
val_items = load_split(DATASET, "validation")
train_items = load_split(DATASET, "train")  # 286 itens: os que têm reflexão gerada

processed_path = ROOT / "data" / "processed" / "arc" / "validation.jsonl"
processed_items = read_jsonl(processed_path)
assert [i["uid"] for i in processed_items] == [i["uid"] for i in val_items], (
    "data/processed/arc/validation.jsonl e data/splits/arc/validation.jsonl divergiram — "
    "investigue antes de continuar."
)

print(f"validação: {len(val_items)} itens   |   treino (com reflexão): {len(train_items)} itens")
pd.DataFrame([
    {
        "uid": i["uid"],
        "pergunta": i["question"][:80] + ("..." if len(i["question"]) > 80 else ""),
        "n_opções": i["num_choices"],
        "gabarito": i["answerKey"],
    }
    for i in val_items[:5]
])


## 2. Prompt de baseline (sem reflexão) — editável

Cópia local de `ANSWER_PROMPT`/`build_answer_prompt` de `rmcq/common.py`. Edite aqui à
vontade; a célula de verificação logo abaixo só avisa se isto divergir do pacote, nunca
trava a execução.


In [ ]:
ANSWER_PROMPT = """You are answering a multiple-choice question.

Question: {question}

Options:
{options}

Instructions:
- Think step by step before answering.
- Choose exactly one option.
- End your response with this exact line, and nothing after it:
FINAL ANSWER: <letter>"""


def format_options(choices):
    return "\n".join(f"{c['label']}) {c['text']}" for c in choices)


def format_question(item):
    context = item.get("context")
    if context:
        return f"{context.strip()}\n\n{item['question'].strip()}"
    return item["question"].strip()


def build_answer_prompt(item):
    return ANSWER_PROMPT.format(question=format_question(item), options=format_options(item["choices"]))


print(build_answer_prompt(val_items[0]))


In [ ]:
# Verificação de sincronia (não trava, só avisa) -----------------------------
from rmcq import common as _common

_diffs = []
if ANSWER_PROMPT != _common.ANSWER_PROMPT:
    _diffs.append("ANSWER_PROMPT")
if build_answer_prompt(val_items[0]) != _common.build_answer_prompt(val_items[0]):
    _diffs.append("build_answer_prompt(...)")

if _diffs:
    warnings.warn(f"cópias locais divergem de rmcq.common em: {_diffs} (pode ser intencional)")
else:
    print("prompt local idêntico a rmcq.common.ANSWER_PROMPT nesta execução.")


## 3. Baseline novo: validação sem reflexão

Não existe ainda — `results/baseline/{modelo}/arc_validation.jsonl` nunca foi gerado (só
`arc_train.jsonl` e `arc_test.jsonl`). Geramos aqui, gravando no mesmo layout que o resto do
pipeline usa (`rmcq.data.baseline_path`), então fica retomável do jeito de sempre e reaproveitável
por qualquer outra análise futura.


In [ ]:
def run_baseline_validation(students, limit=None, backend_kind=None):
    params = GenParams.from_config(STUDENT_GEN, seed=SEED)
    stats = {"generated": 0, "elapsed_s": 0.0}

    items = val_items[:limit] if limit else val_items

    for model in students:
        store = JsonlStore(baseline_path(model, DATASET, "validation"))
        done = store.done_keys()
        pending = [i for i in items if i["uid"] not in done]
        if not pending:
            log.info("[%s] baseline de validação já completo (%d itens)", model, len(items))
            continue

        with Timer() as timer, get_backend(model, backend_kind) as backend:
            prompts = [build_answer_prompt(i) for i in pending]
            gens = backend.generate(prompts, params, desc=f"{model} baseline arc/validation")

            records = [
                make_record(
                    item, stage="baseline", condition="no_reflection", student_model=model,
                    prompt=prompt, output=gen.text,
                    prompt_tokens=gen.prompt_tokens, completion_tokens=gen.completion_tokens,
                    latency_s=gen.latency_s, seed=SEED, temperature=params.temperature,
                )
                for item, prompt, gen in zip(pending, prompts, gens)
            ]
            store.append(records)
            stats["generated"] += len(records)
            n_ok = sum(1 for r in records if r.is_correct)
            log.info("[%s] gravado %d, acerto %.1f%%", model, len(records), 100 * n_ok / max(len(records), 1))
        stats["elapsed_s"] += timer.elapsed

    return stats


run_baseline_validation(STUDENTS, limit=RUN_LIMIT, backend_kind=BACKEND_KIND)


In [ ]:
def load_rows(path):
    store = JsonlStore(path)
    return {r["uid"]: r for r in store.read_all()} if store.exists() else {}


baseline_rows = {s: load_rows(baseline_path(s, DATASET, "validation")) for s in STUDENTS}

baseline_summary = pd.DataFrame([
    {"student": s, **accuracy_block(list(rows.values()))}
    for s, rows in baseline_rows.items()
])
baseline_summary[["student", "n", "n_answered", "n_abstained", "n_correct", "accuracy", "accuracy_answered", "format_adherence"]]


## 4. Similaridade treino ↔ validação

O índice oficial do projeto (`results/index/BAAI_bge-large-en-v1.5/arc/`) compara treino
contra **teste**, nunca contra validação — `rmcq.retrieval.build` só conhece esses dois splits.
Por isso calculamos aqui uma similaridade nova, treino×validação, com o **mesmo** embedder,
**mesmo** texto de entrada (contexto + pergunta, sem alternativas — igual a
`rmcq.retrieval._embed_text`) e o mesmo prefixo de consulta do BGE do lado da validação.

Otimização: se o índice oficial já tem os embeddings de treino calculados (`train.npy`), nós
os reaproveitamos em vez de recalcular — é a mesma lógica de "índice compartilhado" que o
pipeline principal usa. Só os embeddings de validação são novos.


In [ ]:
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def needs_query_prefix(embedder_name):
    return "bge" in embedder_name.lower() and "en" in embedder_name.lower()


def embed_text(item):
    """Texto representado no índice: contexto + pergunta, sem as alternativas."""
    context = (item.get("context") or "").strip()
    question = item["question"].strip()
    return f"{context}\n\n{question}".strip() if context else question


FORCE_RECOMPUTE_SIM = False  # True para ignorar qualquer cache e recalcular tudo

SIM_CACHE_DIR = INDEX_DIR / EMBEDDER.replace("/", "_") / DATASET
SIM_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SIM_CACHE_PATH = SIM_CACHE_DIR / "train_validation_sim.npz"

train_by_uid = {i["uid"]: i for i in train_items}
official_index = index_paths(DATASET, EMBEDDER)


In [ ]:
if SIM_CACHE_PATH.exists() and not FORCE_RECOMPUTE_SIM:
    _blob = np.load(SIM_CACHE_PATH, allow_pickle=False)
    sim_matrix = _blob["sim"]
    train_uids = [str(u) for u in _blob["train_uids"]]
    val_uids = [str(u) for u in _blob["val_uids"]]
    print(f"similaridade carregada do cache: {SIM_CACHE_PATH}")

else:
    # Reaproveita o embedding de treino do índice oficial (train × test) quando existir:
    # o embedding de uma questão não depende de contra o que ela vai ser comparada.
    if official_index["train_emb"].exists() and official_index["neighbors"].exists():
        _nb = np.load(official_index["neighbors"], allow_pickle=False)
        _official_train_uids = [str(u) for u in _nb["train_uids"]]
        if set(_official_train_uids) == set(train_by_uid):
            train_emb = np.load(official_index["train_emb"])
            train_uids = _official_train_uids
            print(f"reaproveitando embeddings de treino do índice oficial: {official_index['train_emb']}")
            need_train_embed = False
        else:
            warnings.warn(
                "uids do índice oficial não batem com data/splits/arc/train.jsonl atual; "
                "recalculando embeddings de treino do zero."
            )
            need_train_embed = True
    else:
        need_train_embed = True

    from sentence_transformers import SentenceTransformer

    embedder_model = SentenceTransformer(EMBEDDER, cache_folder=str(HF_HOME), token=hf_token())

    if need_train_embed:
        train_uids = [i["uid"] for i in train_items]
        train_texts = [embed_text(i) for i in train_items]
        train_emb = embedder_model.encode(
            train_texts, batch_size=EMBED_BATCH_SIZE, normalize_embeddings=True,
            show_progress_bar=True, convert_to_numpy=True,
        )

    val_uids = [i["uid"] for i in val_items]
    val_texts = [embed_text(i) for i in val_items]
    if needs_query_prefix(EMBEDDER):
        val_texts = [BGE_QUERY_PREFIX + t for t in val_texts]
    val_emb = embedder_model.encode(
        val_texts, batch_size=EMBED_BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    )

    sim_matrix = val_emb @ train_emb.T  # (n_val, n_train), embeddings normalizados => cosseno
    np.savez_compressed(
        SIM_CACHE_PATH,
        sim=sim_matrix.astype("float32"),
        train_uids=np.array(train_uids),
        val_uids=np.array(val_uids),
    )
    print(f"similaridade calculada e cacheada em: {SIM_CACHE_PATH}")

val_uid_to_row = {u: i for i, u in enumerate(val_uids)}
train_uid_to_col = {u: i for i, u in enumerate(train_uids)}
print(f"sim_matrix: {sim_matrix.shape}  (validação × treino)")


In [ ]:
top1_sim = sim_matrix.max(axis=1)

display(pd.Series(top1_sim, name="top1_similarity").describe(percentiles=[.1, .25, .5, .75, .9, .95]))

fig, ax = plt.subplots()
ax.hist(top1_sim, bins=30)
ax.set_xlabel("similaridade (cosseno) da questão de treino mais próxima")
ax.set_ylabel("nº de questões de validação")
ax.set_title(f"ARC — similaridade top-1 treino↔validação ({EMBEDDER})")
plt.tight_layout()
plt.show()


## 5. Escolha de `k` e do limiar de similaridade

Data-driven: os limiares candidatos vêm dos quartis da distribuição de similaridade top-1
acima (0 = sem filtro nenhum, igual ao comportamento atual do pipeline). Ajuste
`K_GRID`/`THRESHOLD_GRID` livremente — são só listas.

O usuário pediu explicitamente para testar `k=2` (e observar quantas reflexões realmente são
recuperadas quando o limiar corta abaixo de `k`); ele entra no grid por padrão junto com 1 e 3
(os valores já usados/candidatos no resto do projeto, `rmcq.config.K_CANDIDATES`).


In [ ]:
K_GRID = [1, 2, 3]

_quantiles = [0.0, 0.25, 0.5, 0.75]
THRESHOLD_GRID = sorted({0.0} | {round(float(np.quantile(top1_sim, q)), 2) for q in _quantiles})

print(f"K_GRID = {K_GRID}")
print(f"THRESHOLD_GRID = {THRESHOLD_GRID}  (quantis {_quantiles} da similaridade top-1)")
print(
    f"\n{len(STUDENTS)} alunos × {len(DEPTHS)} profundidades × {len(K_GRID)} valores de k × "
    f"{len(THRESHOLD_GRID)} limiares = "
    f"{len(STUDENTS) * len(DEPTHS) * len(K_GRID) * len(THRESHOLD_GRID)} configurações"
)


## 6. Recuperação (k, threshold) — com rastreio de quantas reflexões vêm de fato

`retrieve_neighbors` devolve no máximo `k` vizinhos de treino com similaridade ≥ `threshold`,
restritos aos uids que têm reflexão de fato gravada (`allowed_train_uids`), em ordem
**crescente** de similaridade — é a ordem que `build_notes_block` (seção seguinte) espera: a
nota mais parecida fica por último, imediatamente antes da questão nova, que é onde um decoder
causal atende mais. Se a lista vier vazia, a etapa 7 cai de
volta no prompt de baseline puro, exatamente como pedido.


In [ ]:
def retrieve_neighbors(sims_row, train_uids, k, threshold, allowed_train_uids=None):
    order = np.argsort(-sims_row)  # do mais similar para o menos
    picked = []
    for idx in order:
        sim = float(sims_row[idx])
        if sim < threshold:
            break  # descendente: nada depois disso passa do limiar
        uid = train_uids[idx]
        if allowed_train_uids is not None and uid not in allowed_train_uids:
            continue
        picked.append((uid, sim))
        if len(picked) >= k:
            break
    picked.reverse()  # crescente
    return picked


def reflections_path_local(student, teacher, depth):
    return RESULTS_DIR / "reflections" / f"{student}__{teacher}__{depth}" / f"{DATASET}.jsonl"


def load_reflections_local(student, teacher, depth):
    """Reflexões de treino indexadas por uid, na mesma interface de rmcq.stages.reflect.load_reflections."""
    path = reflections_path_local(student, teacher, depth)
    if not path.exists():
        raise FileNotFoundError(f"reflexões ausentes: {path}")
    return {r["uid"]: r for r in read_jsonl(path)}


# Só self-reflection (professor == aluno) por padrão — é o que foi pedido: cada aluno
# reaproveita as próprias reflexões de treino. Para incluir reflexão externa, basta
# adicionar pares (aluno, professor) diferentes aqui.
TEACHERS_PER_STUDENT = {s: [s] for s in STUDENTS}

REFLECTIONS = {
    (student, teacher, depth): load_reflections_local(student, teacher, depth)
    for student in STUDENTS
    for teacher in TEACHERS_PER_STUDENT[student]
    for depth in DEPTHS
}
print(f"{len(REFLECTIONS)} conjuntos de reflexão carregados: {list(REFLECTIONS.keys())}")


### Os prompts que geraram estas reflexões (histórico, só leitura)

As reflexões em `results/reflections/` já existem; este notebook não as regera. Os quatro
prompts abaixo são os que as produziram — estão aqui porque metade do diagnóstico é olhar
para o texto que o professor foi instruído a escrever e perguntar se ele **transfere** para
outra questão. Editar esta célula não muda nada: para mudar a reflexão é preciso reescrever
`REFLECTION_PROMPTS` em `rmcq/common.py`, apagar o diretório correspondente e rodar
`python -m rmcq reflect` de novo.


In [ ]:
for (_depth, _persp), _text in sorted(_common.REFLECTION_PROMPTS.items()):
    if _depth not in DEPTHS:
        continue
    print("=" * 78)
    print(f"REFLECTION_PROMPTS[({_depth!r}, {_persp!r})]   "
          f"— {'autorreflexão (aluno == professor)' if _persp == 'student' else 'professor externo'}")
    print("=" * 78)
    print(_text)
    print()


In [ ]:
# Pré-visualização SEM chamar nenhum modelo: quantas reflexões cada (k, threshold) realmente
# entrega, em média, e em que fração dos itens de validação a busca não encontra NADA acima do
# limiar (cai no baseline). Vale olhar isto antes de gastar GPU na grade completa.

coverage_rows = []
for student in STUDENTS:
    for teacher in TEACHERS_PER_STUDENT[student]:
        for depth in DEPTHS:
            refl = REFLECTIONS[(student, teacher, depth)]
            allowed = {u for u, r in refl.items() if r.get("reflection_text")}
            for k in K_GRID:
                for threshold in THRESHOLD_GRID:
                    counts = np.array([
                        len(retrieve_neighbors(sim_matrix[val_uid_to_row[u]], train_uids, k, threshold, allowed))
                        for u in val_uids
                    ])
                    coverage_rows.append({
                        "student": student, "teacher": teacher, "depth": depth,
                        "k": k, "threshold": threshold,
                        "mean_retrieved": counts.mean(),
                        "pct_full_k": (counts == k).mean(),
                        "pct_zero_fallback_baseline": (counts == 0).mean(),
                    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.round(3)


## 7. Prompt de avaliação com reflexões recuperadas — editável

**Tudo que entra no prompt está nas duas células abaixo.** Nada de prompt é importado de
`rmcq/common.py`: os textos ficam na primeira célula (é o que você edita) e as
transformações aplicadas a cada reflexão ficam na segunda. A célula de verificação compara
o resultado com o `rmcq.common.build_eval_prompt` versão `v2` e só **avisa** se divergir —
divergir é o ponto do notebook.

Layout `v2`, reescrito para modelos pequenos (phi4-mini, llama3-8b). Cada decisão veio de um
número medido nas reflexões deste repositório:

| decisão | por quê |
|---|---|
| enquadramento primeiro, questão e formato de saída por último | no layout antigo as lições abriam o prompt: o modelo lia até 615 palavras de crítica antes de saber que ia responder uma questão, e a instrução de formato ficava a ~2.000 tokens do início |
| `<notes>` / `<note id="i">` em vez de `[Lesson 1]` + `---` | sem delimitador duro o enunciado de origem compete com o enunciado a responder |
| enunciado de origem **com contexto** (`format_question`), 600 chars, corte em fronteira de frase | a similaridade da seção 4 é calculada sobre contexto + pergunta; injetar só a pergunta deixa a nota como conselho sem referente |
| corte de cada nota em 120 palavras, **pela cabeça** | `complex` tem ~615 palavras de mediana; com k=3 são ~1.850 palavras de referência para uma questão de ~60. A parte que transfere ("Lesson: …", "How to improve: …") fica na **cauda** |
| letras de alternativa → `[letter]` | 50% das reflexões `complex` e 42% das de autorreflexão citam uma letra vinda de **outra** questão, ancorando o aluno numa alternativa da questão nova |

Sem nenhuma reflexão recuperada, o prompt cai **byte a byte** no `ANSWER_PROMPT` da
seção 2 — não existe um terceiro formato para "k pedido mas nada recuperado".


In [ ]:
# =========================================================================
# OS TEXTOS DO PROMPT — é isto que você edita
# =========================================================================

NOTES_HEADER = """You are answering a multiple-choice question.

First, some notes from earlier attempts at OTHER questions. They are reference material about how to reason. None of them is about the question below, and none of them contains its answer.

<notes>
{notes}
</notes>

"""

NOTE_BLOCK = """<note id="{i}">
{body}
</note>"""

NOTE_SOURCE = 'This note was written about a different question: "{source_question}"'

NOTE_OUTCOME = {
    True: "The earlier answer to that question was correct.",
    False: "The earlier answer to that question was incorrect.",
}

EVAL_TAIL = """Question: {question}

Options:
{options}

Instructions:
- The notes are advice on how to reason, nothing more. The correct option here may be a different letter than any letter mentioned in a note.
- Think step by step before answering.
- Choose exactly one option.
- End your response with this exact line, and nothing after it:
FINAL ANSWER: <letter>"""


# --- knobs: cada um é uma ablação, ligue e desligue à vontade ---------------
INJECT_SOURCE_QUESTION = True    # o enunciado de origem entra junto de cada nota
TAG_SOURCE_OUTCOME = True        # "the earlier answer to that question was correct/incorrect"
NOTE_MAX_WORDS = 120             # 0 desliga o corte. 120 ~ tamanho de uma reflexão `simple`
NEUTRALIZE_LETTERS = True        # "you selected D" -> "you selected [letter]"
SOURCE_QUESTION_MAX_CHARS = 600  # 0 ou None desliga o corte do enunciado de origem


In [ ]:
# =========================================================================
# AS TRANSFORMAÇÕES APLICADAS A CADA REFLEXÃO ANTES DE ELA VIRAR UMA <note>
# =========================================================================
# Cópia local de rmcq/common.py, aqui para poder ser editada — a célula de
# verificação abaixo avisa quando o resultado deixa de bater com o oficial.
import re

_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
_BULLET = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s")

# "option D", "answer (B)", "chose C" — sempre letra MAIÚSCULA isolada, então o
# artigo "a" e palavras comuns não são atingidos.
_LETTER_MENTIONS = (
    re.compile(r"\b((?:option|choice|answer|alternative|response)s?\s+)\(?([A-H])\)?(?![\w-])"),
    re.compile(r"\b((?:chose|choose|chosen|selected|select|picked|pick|answered)\s+)\(?([A-H])\)?(?![\w-])"),
    re.compile(r"(\s)\(([A-H])\)(?![\w-])"),
)
# Continuação de enumeração: "options [letter], C, and D".
_LETTER_ENUM = re.compile(r"(\[letter\])(,?\s+(?:and|or)\s+|,\s*)([A-H])(?![\w-])")
_LETTER_QUOTED = re.compile(r"([\"“\'])([A-H])([\"”\'])")


def neutralize_option_letters(text):
    """
    "you selected D" -> "you selected [letter]".

    A reflexão foi escrita sobre OUTRA questão, onde "D" era outra coisa. A frase
    continua legível; só perde a âncora numa letra que não tem nada a ver com a
    questão nova. NÃO cobre letra solta sem pista ("arguing that A could be
    possible"): separar isso do artigo "A" daria falso positivo em texto comum.
    """
    for pattern in _LETTER_MENTIONS:
        text = pattern.sub(r"\1[letter]", text)
    text = _LETTER_QUOTED.sub(r"\1[letter]\3", text)
    while True:
        text, n = _LETTER_ENUM.subn(r"\1\2[letter]", text)
        if not n:
            return text


def _text_units(text):
    """Linhas e frases da reflexão, na ordem, sem vazios — a unidade de corte."""
    units = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in _SENTENCE_END.split(line) if p.strip()]
        units.extend(parts or [line])
    return units


def compact_reflection(text, max_words=None):
    """
    Limita a reflexão a `max_words` cortando pela CABEÇA.

    A cabeça narra a questão de origem ("Your reasoning centered on a causal
    chain: government policies increased demand...") e a cauda traz o que
    transfere ("Use the negation test for necessary assumptions..."). Cortar
    pela cauda joga fora exatamente a parte reaproveitável.
    """
    max_words = NOTE_MAX_WORDS if max_words is None else max_words
    text = (text or "").strip()
    if max_words <= 0 or len(text.split()) <= max_words:
        return text

    kept, total = [], 0
    for unit in reversed(_text_units(text)):
        n = len(unit.split())
        if kept and total + n > max_words:
            break
        kept.append(unit)
        total += n
    kept.reverse()
    if not kept:
        return " ".join(text.split()[:max_words])
    out = ""
    for unit in kept:
        sep = "\n" if out and _BULLET.match(unit) else (" " if out else "")
        out += sep + unit
    return "(...) " + out


def format_source_question(text, max_chars=None):
    """Enunciado de origem em uma linha, cortado em fronteira de frase."""
    max_chars = SOURCE_QUESTION_MAX_CHARS if max_chars is None else max_chars
    text = " ".join((text or "").split())
    if not max_chars or len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    stop = max(cut.rfind(". "), cut.rfind("? "), cut.rfind("! "))
    return (cut[: stop + 1] if stop > max_chars // 2 else cut.rstrip()) + " (...)"


# --- montagem --------------------------------------------------------------

def build_notes_block(reflections, source_questions=None, source_was_correct=None):
    """As k notas, na ordem em que chegam (similaridade CRESCENTE)."""
    if not reflections:
        return ""

    notes = []
    for i, reflection in enumerate(reflections, start=1):
        lines = []
        if INJECT_SOURCE_QUESTION and source_questions:
            lines.append(NOTE_SOURCE.format(source_question=format_source_question(source_questions[i - 1])))
        if TAG_SOURCE_OUTCOME and source_was_correct:
            outcome = NOTE_OUTCOME.get(source_was_correct[i - 1])
            if outcome:
                lines.append(outcome)

        text = compact_reflection(reflection)
        if NEUTRALIZE_LETTERS:
            text = neutralize_option_letters(text)
        lines.append(text)

        notes.append(NOTE_BLOCK.format(i=i, body="\n".join(lines)))

    return NOTES_HEADER.format(notes="\n\n".join(notes))


def build_eval_prompt(item, reflections, source_questions=None, source_was_correct=None):
    # Sem notas recuperadas o prompt é byte a byte o baseline da seção 2 — é
    # isso que torna a linha de fallback comparável ao baseline.
    if not reflections:
        return build_answer_prompt(item)

    return build_notes_block(reflections, source_questions, source_was_correct) + EVAL_TAIL.format(
        question=format_question(item),
        options=format_options(item["choices"]),
    )


In [ ]:
# Verificação de sincronia (não trava, só avisa) -----------------------------
# O kernel pode ter importado rmcq.common ANTES de o arquivo mudar — o Python
# cacheia o módulo, e a comparação abaixo levantaria TypeError com a assinatura
# antiga. Recarregar aqui evita ter que reiniciar o kernel a cada edição.
import importlib

from rmcq import common as _common
_common = importlib.reload(_common)

_sample = val_items[0]
_refl = REFLECTIONS[(STUDENTS[0], STUDENTS[0], "simple")]
_allowed = {u for u, r in _refl.items() if r.get("reflection_text")}
_picked = retrieve_neighbors(sim_matrix[val_uid_to_row[_sample["uid"]]], train_uids, k=2, threshold=0.0, allowed_train_uids=_allowed)
_texts = [_refl[u]["reflection_text"] for u, _ in _picked]
# format_question, não ["question"]: com o contexto, que é sobre o que a
# similaridade da seção 4 foi calculada.
_srcq = [format_question(train_by_uid[u]) for u, _ in _picked]
_srcc = [_refl[u].get("extra", {}).get("source_was_correct") for u, _ in _picked]

_local_prompt = build_eval_prompt(_sample, _texts, _srcq, _srcc)

try:
    _official_prompt = _common.build_eval_prompt(_sample, _texts, _srcq, _srcc, version="v2")
except TypeError:
    # rmcq/common.py ainda sem o prompt v2 (checkout anterior a 26/08/2026).
    _official_prompt = None
    warnings.warn("rmcq.common deste ambiente não conhece o prompt v2; comparação pulada.")

if _official_prompt is None:
    pass
elif _local_prompt != _official_prompt:
    warnings.warn("prompt local diverge de rmcq.common (v2) — intencional se você editou a célula acima")
else:
    print("prompt de avaliação local idêntico a rmcq.common (v2) nesta execução.")

print(f"fallback sem nota == baseline byte a byte: {build_eval_prompt(_sample, []) == build_answer_prompt(_sample)}")

print("\n" + "=" * 78)
print("EXEMPLO — o prompt exato que o aluno recebe (k=2, sem filtro de limiar)")
print("=" * 78 + "\n")
print(_local_prompt)


### O que cada knob faz, no texto de verdade

A célula abaixo não chama modelo nenhum: ela mostra o efeito de cada knob sobre as
reflexões deste dataset e o custo em tokens de entrada. É o lugar de decidir os valores
antes de gastar GPU na grade.


In [ ]:
# Efeito de cada knob, medido nas reflexões deste dataset -------------------
_demo_texts = [r["reflection_text"] for r in _refl.values() if r.get("reflection_text")]

print("tamanho das reflexões (palavras), por profundidade:")
for _depth in DEPTHS:
    _t = [r["reflection_text"] for r in REFLECTIONS[(STUDENTS[0], STUDENTS[0], _depth)].values() if r.get("reflection_text")]
    _w = sorted(len(x.split()) for x in _t)
    _cortadas = sum(1 for x in _t if NOTE_MAX_WORDS and len(x.split()) > NOTE_MAX_WORDS)
    _com_letra = sum(1 for x in _t if neutralize_option_letters(x) != x)
    print(
        f"  {_depth:8s} mediana {_w[len(_w)//2]:4d}  máx {_w[-1]:4d}  |  "
        f"acima de NOTE_MAX_WORDS={NOTE_MAX_WORDS}: {_cortadas}/{len(_t)}  |  "
        f"citam letra de alternativa: {_com_letra}/{len(_t)}"
    )

# Uma reflexão longa, antes e depois do corte -------------------------------
_long = max(_demo_texts, key=lambda x: len(x.split()))
print("\n" + "-" * 78)
print(f"CORTE — reflexão mais longa: {len(_long.split())} palavras")
print("-" * 78)
print("ANTES (primeiras 60 palavras):", " ".join(_long.split()[:60]), "...")
print("\nDEPOIS:", compact_reflection(_long))

# Neutralização de letra ----------------------------------------------------
_with_letter = next((t for t in _demo_texts if neutralize_option_letters(t) != t), None)
if _with_letter:
    import difflib as _difflib
    print("\n" + "-" * 78)
    print("NEUTRALIZAÇÃO DE LETRA — trechos alterados")
    print("-" * 78)
    _a, _b = _with_letter.split(), neutralize_option_letters(_with_letter).split()
    for _op, _i1, _i2, _j1, _j2 in _difflib.SequenceMatcher(None, _a, _b).get_opcodes():
        if _op != "equal":
            print("  ", " ".join(_a[max(0, _i1 - 6):_i2 + 4]), " ->  ", " ".join(_b[max(0, _j1 - 6):_j2 + 4]))

# Custo em tokens de entrada, por configuração ------------------------------
print("\n" + "-" * 78)
print("CUSTO — palavras do prompt de avaliação (proxy de tokens de entrada)")
print("-" * 78)
_base_len = len(build_answer_prompt(_sample).split())
print(f"  baseline (sem nota): {_base_len} palavras")
for _k in K_GRID:
    _p = retrieve_neighbors(sim_matrix[val_uid_to_row[_sample["uid"]]], train_uids, _k, 0.0, _allowed)
    _tx = [_refl[u]["reflection_text"] for u, _ in _p]
    _sq = [format_question(train_by_uid[u]) for u, _ in _p]
    _sc = [_refl[u].get("extra", {}).get("source_was_correct") for u, _ in _p]
    _cortado = len(build_eval_prompt(_sample, _tx, _sq, _sc).split())
    _sem_corte = NOTE_MAX_WORDS
    globals()["NOTE_MAX_WORDS"] = 0
    _bruto = len(build_eval_prompt(_sample, _tx, _sq, _sc).split())
    globals()["NOTE_MAX_WORDS"] = _sem_corte
    print(f"  k={_k}: {_cortado} palavras (sem o corte seriam {_bruto}, ou {_bruto / max(_base_len, 1):.1f}x o baseline)")


## 8. A grade de experimentos

Cada configuração é `(aluno, professor, profundidade, k, threshold)`. Com professor == aluno
(self-reflection) por padrão. Os resultados vão para
`results/diagnostics/{aluno}__{professor}__{profundidade}__k{k}__t{threshold}/arc_validation.jsonl`
— uma árvore nova, separada de `results/eval/` (que é sobre o split de teste e não tem eixo de
threshold), para não colidir com nada do pipeline oficial.


In [ ]:
GRID = [
    {"student": s, "teacher": t, "depth": d, "k": k, "threshold": thr}
    for s in STUDENTS
    for t in TEACHERS_PER_STUDENT[s]
    for d in DEPTHS
    for k in K_GRID
    for thr in THRESHOLD_GRID
]

n_items = len(val_items) if not RUN_LIMIT else min(RUN_LIMIT, len(val_items))
print(f"{len(GRID)} configurações × {n_items} itens = {len(GRID) * n_items:,} gerações no total")


def diag_tag(student, teacher, depth, k, threshold):
    t_tag = f"t{threshold:.2f}".replace(".", "p")
    return f"{student}__{teacher}__{depth}__k{k}__{t_tag}"


def diag_eval_path(student, teacher, depth, k, threshold):
    d = RESULTS_DIR / "diagnostics" / diag_tag(student, teacher, depth, k, threshold)
    d.mkdir(parents=True, exist_ok=True)
    return d / "arc_validation.jsonl"


## 9. Rodando a grade

Agrupado por aluno (uma carga de modelo por aluno, igual à etapa 4 do pipeline principal —
`rmcq.stages.evaluate`). Retomável: uma configuração já gravada não é refeita, então interromper
e rodar de novo (inclusive em outra sessão) só completa o que falta.

Com `SMOKE_TEST = True` (célula de setup), isto roda em segundos com o `StubBackend`. Para a
rodada de verdade, mude `SMOKE_TEST = False`, reinicie a partir da célula de setup (para
recarregar `BACKEND_KIND`/`RUN_LIMIT`) e rode esta célula — pode levar horas, é esperado.


In [ ]:
def run_diagnostic_grid(grid, limit=None, backend_kind=None):
    from collections import defaultdict

    items = val_items[:limit] if limit else val_items
    items_by_uid = {i["uid"]: i for i in items}
    use_uids = list(items_by_uid)

    by_student = defaultdict(list)
    for cfg in grid:
        by_student[cfg["student"]].append(cfg)

    params = GenParams.from_config(STUDENT_GEN, seed=SEED)
    stats = {"generated": 0, "elapsed_s": 0.0}

    for student, cfgs in by_student.items():
        # Resolve pendências ANTES de carregar o modelo — igual a
        # rmcq.stages.evaluate.run. Sem isto, um rerun que já completou tudo
        # para este aluno carregaria o modelo à toa.
        work = []
        for cfg in cfgs:
            path = diag_eval_path(cfg["student"], cfg["teacher"], cfg["depth"], cfg["k"], cfg["threshold"])
            done = JsonlStore(path).done_keys()
            pending_uids = [u for u in use_uids if u not in done]
            if pending_uids:
                work.append((cfg, path, pending_uids))

        if not work:
            log.info("[aluno=%s] nada a fazer", student)
            continue

        with Timer() as timer, get_backend(student, backend_kind) as backend:
            for cfg, path, pending_uids in work:
                teacher, depth, k, threshold = cfg["teacher"], cfg["depth"], cfg["k"], cfg["threshold"]
                store = JsonlStore(path)

                refl = REFLECTIONS[(student, teacher, depth)]
                allowed = {u for u, r in refl.items() if r.get("reflection_text")}
                condition = "self_reflection" if student == teacher else "external_reflection"

                prompts, metas = [], []
                for uid in pending_uids:
                    picked = retrieve_neighbors(sim_matrix[val_uid_to_row[uid]], train_uids, k, threshold, allowed)
                    texts = [refl[u]["reflection_text"] for u, _ in picked]
                    # format_question: com o contexto, que é sobre o que a
                    # similaridade da seção 4 foi calculada.
                    src_qs = [format_question(train_by_uid[u]) for u, _ in picked]
                    src_correct = [refl[u].get("extra", {}).get("source_was_correct") for u, _ in picked]

                    item = items_by_uid[uid]
                    prompts.append(build_eval_prompt(item, texts, src_qs, src_correct))
                    metas.append((item, [u for u, _ in picked], [s for _, s in picked]))

                # Sem system=: o prompt v2 é autocontido e o baseline da seção 3
                # também rodou sem system. Um system só na condição de reflexão
                # mediria prompt + reflexão de uma vez.
                gens = backend.generate(
                    prompts, params,
                    desc=f"{student} diag {teacher}/{depth}/k{k}/t{threshold:.2f}",
                )

                records = [
                    make_record(
                        item, stage="eval_diagnostic", condition=condition, student_model=student,
                        teacher_model=teacher, prompt=prompt, output=gen.text,
                        reflection_depth=depth,
                        reflection_perspective=("student" if student == teacher else "teacher"),
                        retrieved_uids=used_uids, retrieved_similarities=[round(s, 6) for s in sims],
                        k=k, prompt_tokens=gen.prompt_tokens, completion_tokens=gen.completion_tokens,
                        latency_s=gen.latency_s, seed=SEED, temperature=params.temperature,
                        extra={
                            "threshold": threshold,
                            "top1_similarity": max(sims) if sims else None,
                            "mean_similarity": (sum(sims) / len(sims)) if sims else None,
                            "n_reflections_injected": len(used_uids),
                            "fallback_to_baseline": len(used_uids) == 0,
                            "eval_prompt_version": "v2-local",
                            "note_max_words": NOTE_MAX_WORDS,
                            "neutralize_letters": NEUTRALIZE_LETTERS,
                            "inject_source_question": INJECT_SOURCE_QUESTION,
                            "tag_source_outcome": TAG_SOURCE_OUTCOME,
                        },
                    )
                    for (item, used_uids, sims), prompt, gen in zip(metas, prompts, gens)
                ]
                store.append(records)
                stats["generated"] += len(records)

                n_ok = sum(1 for r in records if r.is_correct)
                short = sum(1 for r in records if len(r.retrieved_uids) < k)
                log.info(
                    "  [%s] %s/%s/k%d/t%.2f: %d respostas, acerto %.1f%%, %d com menos de k reflexões",
                    student, teacher, depth, k, threshold, len(records),
                    100 * n_ok / max(len(records), 1), short,
                )
        stats["elapsed_s"] += timer.elapsed

    return stats


run_diagnostic_grid(GRID, limit=RUN_LIMIT, backend_kind=BACKEND_KIND)


## 10. Consolidação

Acurácia e **reflection utility** (`rmcq.stages.analyze.utility`, a mesma métrica usada em
`analysis.ipynb`: taxa de virada errado→certo menos certo→errado, contra o mesmo baseline) de
cada configuração da grade. `mcnemar_p` é o teste de McNemar exato sobre os pares
discordantes (`wrong_to_right` vs `right_to_wrong`) — o teste certo aqui porque baseline e
condição respondem os MESMOS itens de validação.


In [ ]:
records = []
for cfg in GRID:
    path = diag_eval_path(**cfg)
    rows = load_rows(path)
    if not rows:
        continue

    base = baseline_rows[cfg["student"]]
    u = utility(base, rows)
    acc = accuracy_block(list(rows.values()))
    n_retrieved = [
        (r.get("extra") or {}).get("n_reflections_injected", len(r.get("retrieved_uids") or []))
        for r in rows.values()
    ]

    records.append({
        **cfg,
        **u,
        "accuracy": acc["accuracy"],
        "accuracy_answered": acc["accuracy_answered"],
        "mean_retrieved": float(np.mean(n_retrieved)) if n_retrieved else 0.0,
        "pct_fallback_baseline": float(np.mean([n == 0 for n in n_retrieved])) if n_retrieved else None,
    })

summary_df = pd.DataFrame(records)

try:
    from scipy.stats import binomtest

    def _mcnemar_p(row):
        b, c = row["wrong_to_right"], row["right_to_wrong"]
        return 1.0 if b + c == 0 else binomtest(min(b, c), b + c, 0.5).pvalue

    summary_df["mcnemar_p"] = summary_df.apply(_mcnemar_p, axis=1)
except ImportError:
    warnings.warn("scipy indisponível: mcnemar_p não calculado")
    summary_df["mcnemar_p"] = None

summary_df.sort_values("utility", ascending=False).round(4)


### Tabela: configurações que superaram o baseline

`utility > 0` significa que, líquido de retrocessos, a reflexão consertou mais itens do que
estragou nesta configuração. `delta_accuracy` é a mesma coisa expressa como diferença de
acurácia bruta.


In [ ]:
wins = summary_df[summary_df["utility"] > 0].sort_values("utility", ascending=False)
cols = [
    "student", "teacher", "depth", "k", "threshold", "n_shared",
    "mean_retrieved", "pct_fallback_baseline",
    "baseline_accuracy", "condition_accuracy", "delta_accuracy", "utility",
    "wrong_to_right", "right_to_wrong", "mcnemar_p",
]
wins_display = wins[[c for c in cols if c in wins.columns]].round(4)

print(f"{len(wins_display)} de {len(summary_df)} configurações superaram o baseline (utility > 0)")
wins_display


In [ ]:
DIAG_DIR = RESULTS_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(DIAG_DIR / "summary_arc_validation.csv", index=False)
wins_display.to_csv(DIAG_DIR / "wins_arc_validation.csv", index=False)
print(f"gravado: {DIAG_DIR / 'summary_arc_validation.csv'}")
print(f"gravado: {DIAG_DIR / 'wins_arc_validation.csv'}")


## 11. Gráficos

### Acurácia vs. limiar de similaridade, por k (linha horizontal = baseline)

In [ ]:
fig, axes = plt.subplots(len(STUDENTS), len(DEPTHS), figsize=(11, 4 * len(STUDENTS)), sharey=True)
axes = np.atleast_2d(axes)

for i, student in enumerate(STUDENTS):
    base_acc = accuracy_block(list(baseline_rows[student].values()))["accuracy"]
    for j, depth in enumerate(DEPTHS):
        ax = axes[i, j]
        sub = summary_df[(summary_df["student"] == student) & (summary_df["depth"] == depth)]
        for k in sorted(sub["k"].unique()):
            line = sub[sub["k"] == k].sort_values("threshold")
            ax.plot(line["threshold"], line["accuracy"], marker="o", label=f"k={k}")
        ax.axhline(base_acc, color="black", linestyle="--", linewidth=1, label="baseline")
        ax.set_title(f"{student} — {depth}")
        ax.set_xlabel("limiar de similaridade")
        if j == 0:
            ax.set_ylabel("acurácia")
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


### Mapa de calor: delta de acurácia (condição − baseline) por k × limiar

In [ ]:
fig, axes = plt.subplots(1, len(STUDENTS) * len(DEPTHS), figsize=(5 * len(STUDENTS) * len(DEPTHS), 4))
axes = np.atleast_1d(axes)

panel = 0
for student in STUDENTS:
    for depth in DEPTHS:
        ax = axes[panel]
        panel += 1
        sub = summary_df[(summary_df["student"] == student) & (summary_df["depth"] == depth)]
        pivot = sub.pivot(index="k", columns="threshold", values="delta_accuracy")
        if pivot.dropna(how="all").empty:
            ax.set_title(f"{student} — {depth} (sem dados)")
            ax.axis("off")
            continue
        # nan-aware: uma grade rodada parcialmente (retomada depois) pode ter buracos.
        vmax = max(np.nanmax(np.abs(pivot.values)), 1e-6)
        im = ax.imshow(pivot.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{t:.2f}" for t in pivot.columns], rotation=45)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel("limiar")
        ax.set_ylabel("k")
        ax.set_title(f"{student} — {depth}")
        for (yi, xi), val in np.ndenumerate(pivot.values):
            ax.text(xi, yi, f"{val:+.3f}", ha="center", va="center", fontsize=8)
        fig.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()


### Transferability: utility × similaridade (sem filtro de limiar, `threshold=0.0`)

A mesma pergunta do paper: a utility decai com a distância semântica entre a questão nova e a
que gerou a reflexão? Usa `rmcq.stages.analyze.transferability`, as mesmas faixas de
similaridade (`SIM_BINS`) do resto do projeto.


In [ ]:
fig, axes = plt.subplots(len(STUDENTS), len(DEPTHS), figsize=(11, 4 * len(STUDENTS)), sharey=True)
axes = np.atleast_2d(axes)

for i, student in enumerate(STUDENTS):
    base = baseline_rows[student]
    for j, depth in enumerate(DEPTHS):
        ax = axes[i, j]
        # k mais alto disponível, sem filtro de limiar: mais chance de cada faixa de
        # similaridade ter itens suficientes para a métrica não ficar vazia.
        k_ref = max(K_GRID)
        rows = load_rows(diag_eval_path(student, student, depth, k_ref, 0.0))
        if not rows:
            continue
        bands = transferability(base, rows, SIM_BINS)
        band_df = pd.DataFrame(bands)
        ax.bar(band_df["sim_bin"], band_df["utility"])
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(f"{student} — {depth} (k={k_ref}, threshold=0.0)")
        ax.set_xlabel("similaridade top-1")
        ax.tick_params(axis="x", rotation=20)
        if j == 0:
            ax.set_ylabel("utility")

plt.tight_layout()
plt.show()


### Quantas reflexões o limiar realmente entrega (a pergunta de rastreio pedida)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (student, depth), grp in coverage_df.groupby(["student", "depth"]):
    for k in sorted(grp["k"].unique()):
        line = grp[grp["k"] == k].sort_values("threshold")
        ax.plot(line["threshold"], line["mean_retrieved"], marker="o", label=f"{student}/{depth}, k={k}")

ax.set_xlabel("limiar de similaridade")
ax.set_ylabel("nº médio de reflexões recuperadas por item")
ax.set_title("Recuperação efetiva vs. k pedido")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 12. Conclusões

Preencher depois de rodar a grade completa (não o smoke test):

- Alguma configuração bateu o baseline de forma consistente (`utility > 0` e `mcnemar_p`
  baixo) nos dois modelos, ou o ganho é específico de um aluno/profundidade?
- O padrão de `pct_fallback_baseline` (seção 6/11) explica os resultados? Se o limiar mais
  alto quase nunca recupera nada, a configuração "vence" só porque vira baseline disfarçado —
  vale checar isso antes de comemorar um ganho.
- O gráfico de transferability (seção 11) mostra utility decaindo com a similaridade, como no
  paper original, ou fica achatado/negativo em toda faixa — sinal de que a reflexão recuperada
  não está transferindo nada, mesmo nos casos "fáceis"?
- Este notebook cobre só self-reflection. Se a conclusão for "nem nos casos mais parecidos a
  reflexão ajuda", o próximo passo natural é repetir com reflexão externa
  (`TEACHERS_PER_STUDENT`) para saber se o problema é da reflexão em si ou específico de cada
  modelo refletir sobre si mesmo.
